### CS 378: Automated Planning for Robots
# HW 3: Modeling Temporal Planning Problems in Unified Planning Framework
### Spring 2026
### Lecturer: Erez Karpas
### TA: Talal Ayman

In this homework assignment you will adapt the planning problem from HW1 to include action durations and allow robot to operate concurrently, by modeling the problem as a temporal planning problem.

Students: **Akshita Santra and Muyang Zhou**

As in HW1, you will use the [Unified Planning Framework](https://github.com/aiplan4eu/unified-planning), so the first step is to make sure the unified planning framework is installed.

We must also make sure to import the package.

In [1]:
%pip install unified-planning[tamer]

import unified_planning
from unified_planning.shortcuts import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 1.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [up-tamer]
Note: you may need to restart the kernel to use updated packages.


We will now create a demo temporal planning problem - the match cellar problem we saw in class.

In [2]:
Match = UserType('Match')
Fuse = UserType('Fuse')

handfree = Fluent('handfree')
light = Fluent('light')
match_used = Fluent('match_used', BoolType(), m=Match)
fuse_mended = Fluent('fuse_mended', BoolType(), f=Fuse)

problem = Problem('MatchCellar')

problem.add_fluent(handfree)
problem.add_fluent(light)
problem.add_fluent(match_used, default_initial_value=False)
problem.add_fluent(fuse_mended, default_initial_value=False)

problem.set_initial_value(light, False)
problem.set_initial_value(handfree, True)

fuses = [Object(f'f{i}', Fuse) for i in range(3)]
matches = [Object(f'm{i}', Match) for i in range(3)]

problem.add_objects(fuses)
problem.add_objects(matches)

light_match = DurativeAction('light_match', m=Match)
m = light_match.parameter('m')
light_match.set_fixed_duration(6)
light_match.add_condition(StartTiming(), Not(match_used(m)))
light_match.add_effect(StartTiming(), match_used(m), True)
light_match.add_effect(StartTiming(), light, True)
light_match.add_effect(EndTiming(), light, False)
problem.add_action(light_match)

mend_fuse = DurativeAction('mend_fuse', f=Fuse)
f = mend_fuse.parameter('f')
mend_fuse.set_fixed_duration(5)
mend_fuse.add_condition(StartTiming(), handfree)
mend_fuse.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), light)
mend_fuse.add_effect(StartTiming(), handfree, False)
mend_fuse.add_effect(EndTiming(), fuse_mended(f), True)
mend_fuse.add_effect(EndTiming(), handfree, True)
problem.add_action(mend_fuse)

for f in fuses:
  problem.add_goal(fuse_mended(f))

print(problem)

problem name = MatchCellar

types = [Match, Fuse]

fluents = [
  bool handfree
  bool light
  bool match_used[m=Match]
  bool fuse_mended[f=Fuse]
]

actions = [
  durative action light_match(Match m) {
    duration = [6, 6]
    conditions = [
      [start]:
        (not match_used(m))
    ]
    effects = [
      start:
        match_used(m) := true:
        light := true:
      end:
        light := false:
    ]
    simulated effects = [
    ]
  }
  durative action mend_fuse(Fuse f) {
    duration = [5, 5]
    conditions = [
      [start]:
        handfree
      [start, end]:
        light
    ]
    effects = [
      start:
        handfree := false:
      end:
        fuse_mended(f) := true:
        handfree := true:
    ]
    simulated effects = [
    ]
  }
]

objects = [
  Match: [m0, m1, m2]
  Fuse: [f0, f1, f2]
]

initial fluents default = [
  bool match_used[m=Match] := false
  bool fuse_mended[f=Fuse] := false
]

initial values = [
  light := false
  handfree := true
]

goals = 

We can now call solve our problem, by using the OneShotPlanner mode.

In [3]:
with OneshotPlanner(name='tamer') as planner:
    result = planner.solve(problem)
    plan = result.plan
    if plan is not None:
        print(f"{planner.name} returned:")
        for start, action, duration in plan.timed_actions:
            print(f"{float(start)}: {action} [{float(duration)}]")
    else:
        print("No plan found.")

NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_8199/1287088451.py`, you are using the following planning engine:
  * Engine name: Tamer
  * Developers:  FBK Tamer Development Team
  * Description: Tamer offers the capability to generate a plan for classical, numerical and temporal problems.
  *              For those kind of problems tamer also offers the possibility of validating a submitted plan.

Tamer returned:
0.0: light_match(m1) [6.0]
0.01: mend_fuse(f1) [5.0]
6.01: light_match(m2) [6.0]
6.02: mend_fuse(f2) [5.0]
12.02: light_match(m0) [6.0]
12.03: mend_fuse(f0) [5.0]


Now that we have made sure everything is working, we can start describing the homework assignment

##  Homework Assignment: Modeling Control of Home Service Robots as a Temporal Planning Problem

### Part1: Modeling
In this homework assignment, you will model a planning problem which can plan for a set of home service robots. This is the same as HW1, except for the addition of action durations.

We have different kinds of robots:
* Mobile manipulator - a mobile robot which can move between rooms, and tidy up a room. However, the mobile manipulator has dirt on its wheelrs, so after a mobile manipulator tidies up a room, it is no longer clean. Moving between rooms takes 30 seconds, tidying up a room takes 300 seconds.
* Mobile vacuum - a mobile robot which can move between rooms, and vacuum the room it is in. A room can only be vacuumed after it has been tidied up. After a room has been vacuumed it is clean. Moving between rooms takes 30 seconds, vacuuming a room takes 400 seconds.

Finally, rooms are connected according to some connectivity map.
The goal is to have all rooms clean and tidy.

Your assignment is to model the given problem using the Unified Planning Framework.
You will need to solve the problem for the following maps:

Map 1:
* 2 connected rooms (rm1 and rm2)
* 1 mobile manipulator, starts in rm1
* 1 vacuum, starts in rm1
* both rooms start not clean and not tidy

Map2:
* 4 rooms - nw, ne, sw, se (4 corners). Each room is connected to two adjacent rooms, and not to the one in the opposite corner (for example, nw is connected to ne and sw, but not to se).
* 1 mobile manipulator, starts in sw
* 1 vacuum, starts in ne
* all rooms start not clean and not tidy

Map3:
* 9 rooms: corridor (which is a room), which is connected to 8 different rooms (rm1 -- rm8) 
* 1 mobile manipulator, starts in corridor
* 1 vacuum, starts in corridor
* rooms rm1 -- rm4 start tidy but not clean, rooms rm5 -- rm8 start clean but not tidy.

In the following code block, you will need to define 3 functions: ``prob1(), prob2(), prob3()`` 
Each of these functions should return a problem object which corresponds to the map described above.

The code block below will call your functions to generate the problem, and then call a planner to solve them and print the solutions.


In [ ]:
def prob1():
    return None

def prob2():
    return None

def prob3():
    return None

The code below will call your functions to generate the problems, and then try to solve them using Pyperplan.

In [ ]:
from unified_planning.engines import PlanGenerationResultStatus

def solve(prob):
    planner = OneshotPlanner(name='tamer')
    result = planner.solve(prob)
    if result.status in [PlanGenerationResultStatus.SOLVED_SATISFICING,
                         PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
        print("SOLVED")
        for start, action, duration in plan.timed_actions:
            print(f"{float(start)}: {action} [{float(duration)}]")
    else:
        print("NOT SOLVED")
    

print("Map1")
solve(prob1())

print("Map2")
solve(prob2())

print("Map3")
solve(prob3())


### Results

**External planner Aries was located online and imported through unified-planning library.**

Map1
SOLVED
0.0: tidy_room(mm1_map1, r1) [300.0]
300.1: move_room_manip(mm1_map1, r1, r2) [30.0]
300.1: clean_room(mv1_map1, r1) [400.0]
330.2: tidy_room(mm1_map1, r2) [300.0]
700.2: move_room_vac(mv1_map1, r1, r2) [30.0]
730.3: clean_room(mv1_map1, r2) [400.0]

Map2
SOLVED
0.0: move_room_manip(mm1_map2, sw, se) [30.0]
0.0: move_room_vac(mv1_map2, ne, nw) [30.0]
30.1: tidy_room(mm1_map2, se) [300.0]
330.2: move_room_manip(mm1_map2, se, sw) [30.0]
360.3: tidy_room(mm1_map2, sw) [300.0]
660.4: move_room_manip(mm1_map2, sw, nw) [30.0]
690.5: tidy_room(mm1_map2, nw) [300.0]
990.6: move_room_manip(mm1_map2, nw, ne) [30.0]
990.6: clean_room(mv1_map2, nw) [400.0]
1020.7: tidy_room(mm1_map2, ne) [300.0]
1390.7: move_room_vac(mv1_map2, nw, ne) [30.0]
1420.8: clean_room(mv1_map2, ne) [400.0]
1820.9: move_room_vac(mv1_map2, ne, se) [30.0]
1851.0: clean_room(mv1_map2, se) [400.0]
2251.1: move_room_vac(mv1_map2, se, sw) [30.0]
2281.2: clean_room(mv1_map2, sw) [400.0]

Map3
SOLVED
0.0: move_room_manip(mm1_map3, corridor, rm5) [30.0]
0.0: move_room_vac(mv1_map3, corridor, rm1) [30.0]
30.1: tidy_room(mm1_map3, rm5) [300.0]
30.1: clean_room(mv1_map3, rm1) [400.0]
330.2: move_room_manip(mm1_map3, rm5, corridor) [30.0]
360.3: move_room_manip(mm1_map3, corridor, rm7) [30.0]
390.4: tidy_room(mm1_map3, rm7) [300.0]
430.2: move_room_vac(mv1_map3, rm1, corridor) [30.0]
460.3: move_room_vac(mv1_map3, corridor, rm3) [30.0]
690.5: move_room_manip(mm1_map3, rm7, corridor) [30.0]
720.6: move_room_manip(mm1_map3, corridor, rm8) [30.0]
750.7: tidy_room(mm1_map3, rm8) [300.0]
1050.8: move_room_manip(mm1_map3, rm8, corridor) [30.0]
1080.9: move_room_manip(mm1_map3, corridor, rm6) [30.0]
1111.0: tidy_room(mm1_map3, rm6) [300.0]
1411.1: clean_room(mv1_map3, rm3) [400.0]
1811.2: move_room_vac(mv1_map3, rm3, corridor) [30.0]
1841.3: move_room_vac(mv1_map3, corridor, rm4) [30.0]
1871.4: clean_room(mv1_map3, rm4) [400.0]
2271.5: move_room_vac(mv1_map3, rm4, corridor) [30.0]
2301.6: move_room_vac(mv1_map3, corridor, rm7) [30.0]
2331.7: clean_room(mv1_map3, rm7) [400.0]
2731.8: move_room_vac(mv1_map3, rm7, corridor) [30.0]
2761.9: move_room_vac(mv1_map3, corridor, rm5) [30.0]
2792.0: clean_room(mv1_map3, rm5) [400.0]
3192.1: move_room_vac(mv1_map3, rm5, corridor) [30.0]
3222.2: move_room_vac(mv1_map3, corridor, rm8) [30.0]
3252.3: clean_room(mv1_map3, rm8) [400.0]
3652.4: move_room_vac(mv1_map3, rm8, corridor) [30.0]
3682.5: move_room_vac(mv1_map3, corridor, rm2) [30.0]
3712.6: clean_room(mv1_map3, rm2) [400.0]
4112.7: move_room_vac(mv1_map3, rm2, corridor) [30.0]
4142.8: move_room_vac(mv1_map3, corridor, rm6) [30.0]
4172.9: clean_room(mv1_map3, rm6) [400.0]

### Part 2: Questions

Please answer each question in the markdown cell below it.
If your answer can be justified using some python code, feel free to also add a code block.

#### Q1

Are the solutions returned by the planner for all problems makespan optimal (meaning, do they take the shortest time possible)?
If so, explain.
If not, give a shorter plan to one of the problems.


#### Answer 1

No, the plan generated for Map 2 is longer than the optimal plan because the mobile vacuum unnecessarily cleans rooms that have not been tidied, meaning it needs to be cleaned again after the mobile manipulator tidies it (since the mobile manipulator dirties the rooms it tidies). A better plan is as follows:

t=0:      tidy_room(mm, sw)                [300s]
t=0:      move_room_vac(ne, nw)            [30s]
t=30:     move_room_vac(nw, sw)            [30s]

t=300:    clean_room(vac, sw)              [400s]
t=300:    move_room_manip(sw, nw)          [30s]
t=330:    tidy_room(mm, nw)               [300s]

t=630:    move_room_manip(nw, ne)          [30s]
t=660:    tidy_room(mm, ne)               [300s]
t=700:    move_room_vac(sw, nw)            [30s]   ← sw done at t=700
t=730:    clean_room(vac, nw)             [400s]

t=960:    move_room_manip(ne, se)          [30s]
t=990:    tidy_room(mm, se)               [300s]

t=1130:   move_room_vac(nw, ne)            [30s]   ← nw done at t=1130
t=1160:   clean_room(vac, ne)             [400s]

t=1290:   move_room_vac(ne, se)            [30s]   ← but ne not done until t=1560
          ✗ need to wait...

t=1560:   move_room_vac(ne, se)            [30s]
t=1590:   clean_room(vac, se)             [400s]

t=1990:   ✓ Done  — Total: ~1990s vs generated 2681s


#### Q2

Above we simplified the problem by assuming it takes 30 seconds to move between rooms. How would you solve the problem if the time to move between rooms was relative to the distance the robot needs to travel? Assume the robot moves at a constant, known, velocity. How would you compute the time to travel between a given pair of rooms? Think about temporal uncertainty, and how it can be avoided.

#### Answer 2

If the robot moves at a known constant velocity v, then travel time is simply: time(ri, rj) = distance(ri, rj)/velocity

So for each pair of rooms: 
travel_time = distance_between_rooms / robot_velocity

To represent the distances in the planning model, we would just need to introduce a numeric fluent for distances:
distance = Fluent('distance', RealType(), rm1=room, rm2=room)

Then initialize it:
map1.set_initial_value(distance(r1, r2), travel_time)
map1.set_initial_value(distance(r2, r1), travel_time)

To define the robot velocity, just set a constant parameter:
VELOCITY = constant given

Then to make the action duration depend on distance, we use duration expressions instead of a fixed duration:
move_room_manip.set_duration(distance(frm, to) / VELOCITY)
move_room_vac.set_duration(distance(frmv, tov) / VELOCITY)

Temporal uncertainty occurs when travel time is not exactly known. Temporal uncertainty can be avoided by:
1. Precomputing exact distances between connected rooms.
2. Using deterministic duration expressions in the planning model.
So every action has a precise duration known at planning time, which keeps the planner determinstic.

# Code

In [ ]:
import unified_planning
from unified_planning.shortcuts import *
from unified_planning.model import StartTiming, EndTiming, ClosedTimeInterval

# Disable credits printing as suggested by the engine warning
unified_planning.shortcuts.get_environment().credits_stream = None

# --------------------
# TYPE DEFINITIONS
# --------------------

room = UserType('room')
mobile_manipulator = UserType('mobile_manipulator')
mobile_vacuum = UserType('mobile_vacuum')


# --------------------
# FLUENT DEFINITIONS
# --------------------

# tidy(rm): room rm is tidy
tidy = Fluent('tidy', rm=room)

# clean(rm): room rm is clean
clean = Fluent('clean', rm=room)

# connected_rooms(rm1, rm2): rm1 and rm2 are connected (movement allowed)
connected = Fluent('connected_rooms', rm1=room, rm2=room)

# current_room_manipulator(r, rm): robot manipulator r is currently located in room rm
current_room_manipulator = Fluent('current_room_manipulator', r=mobile_manipulator, rm=room)

# current_room_vacuum(r, rm): robot vacuum r is currently located in room rm
current_room_vacuum = Fluent('current_room_vacuum', r=mobile_vacuum, rm=room)

# moving_manipulator(r): manipulator r is currently in transit
moving_manipulator = Fluent('moving_manipulator', r=mobile_manipulator)

# moving_vacuum(r): vacuum r is currently in transit
moving_vacuum = Fluent('moving_vacuum', r=mobile_vacuum)


# --------------------
# ACTION: ROBOT MANIPULATOR MOVE BETWEEN ROOMS
# --------------------
move_room_manip = DurativeAction('move_room_manip', r=mobile_manipulator, rm_from=room, rm_to=room)
move_room_manip.set_fixed_duration(30)
rm = move_room_manip.parameter('r')
frm = move_room_manip.parameter('rm_from')
to = move_room_manip.parameter('rm_to')

# Preconditions:
# - robot must be in rm_from at start
# - rm_from and rm_to must be connected
# - robot must not already be moving
move_room_manip.add_condition(StartTiming(), current_room_manipulator(rm, frm))
move_room_manip.add_condition(StartTiming(), connected(frm, to))
move_room_manip.add_condition(StartTiming(), Not(moving_manipulator(rm)))

# Effects:
# - robot leaves rm_from at start
# - robot is marked as moving at start
# - robot arrives in rm_to at end
# - robot is no longer moving at end
move_room_manip.add_effect(StartTiming(), current_room_manipulator(rm, frm), False)
move_room_manip.add_effect(StartTiming(), moving_manipulator(rm), True)
move_room_manip.add_effect(EndTiming(), current_room_manipulator(rm, to), True)
move_room_manip.add_effect(EndTiming(), moving_manipulator(rm), False)


# --------------------
# ACTION: ROBOT VACUUM MOVE BETWEEN ROOMS
# --------------------
move_room_vac = DurativeAction('move_room_vac', r=mobile_vacuum, rm_from=room, rm_to=room)
move_room_vac.set_fixed_duration(30)
rmv = move_room_vac.parameter('r')
frmv = move_room_vac.parameter('rm_from')
tov = move_room_vac.parameter('rm_to')

# Preconditions:
# - robot must be in rm_from at start
# - rm_from and rm_to must be connected
# - robot must not already be moving
move_room_vac.add_condition(StartTiming(), current_room_vacuum(rmv, frmv))
move_room_vac.add_condition(StartTiming(), connected(frmv, tov))
move_room_vac.add_condition(StartTiming(), Not(moving_vacuum(rmv)))

# Effects:
# - robot leaves rm_from at start
# - robot is marked as moving at start
# - robot arrives in rm_to at end
# - robot is no longer moving at end
move_room_vac.add_effect(StartTiming(), current_room_vacuum(rmv, frmv), False)
move_room_vac.add_effect(StartTiming(), moving_vacuum(rmv), True)
move_room_vac.add_effect(EndTiming(), current_room_vacuum(rmv, tov), True)
move_room_vac.add_effect(EndTiming(), moving_vacuum(rmv), False)


# --------------------
# ACTION: TIDY ROOM
# --------------------
tidy_room = DurativeAction('tidy_room', r=mobile_manipulator, rm=room)
tidy_room.set_fixed_duration(300)
tidy_room_robot = tidy_room.parameter('r')
tidy_room_room = tidy_room.parameter('rm')

# Preconditions:
# - manipulator must be in the room at start and must stay for the full duration
# - manipulator must not be mid-move
tidy_room.add_condition(StartTiming(), current_room_manipulator(tidy_room_robot, tidy_room_room))
tidy_room.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), current_room_manipulator(tidy_room_robot, tidy_room_room))
tidy_room.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), Not(moving_manipulator(tidy_room_robot)))

# Effects:
# - room becomes tidy
# - room is no longer clean (tidying may introduce dirt)
tidy_room.add_effect(EndTiming(), tidy(tidy_room_room), True)
tidy_room.add_effect(EndTiming(), clean(tidy_room_room), False)


# --------------------
# ACTION: CLEAN A ROOM
# --------------------
clean_room = DurativeAction('clean_room', r=mobile_vacuum, rm=room)
clean_room.set_fixed_duration(400)
clean_room_robot = clean_room.parameter('r')
clean_room_room = clean_room.parameter('rm')

# Preconditions:
# - vacuum must be in the room at start and must stay for the full duration
# - room must already be tidy
# - vacuum must not be mid-move
clean_room.add_condition(StartTiming(), current_room_vacuum(clean_room_robot, clean_room_room))
clean_room.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), current_room_vacuum(clean_room_robot, clean_room_room))
clean_room.add_condition(StartTiming(), tidy(clean_room_room))
clean_room.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), Not(moving_vacuum(clean_room_robot)))

# Effects:
# - room becomes clean
clean_room.add_effect(EndTiming(), clean(clean_room_room), True)


# --------------------
# Base Problem
# --------------------
problem = Problem('problem')

problem.add_fluent(tidy)
problem.add_fluent(clean)
problem.add_fluent(connected)
problem.add_fluent(current_room_manipulator)
problem.add_fluent(current_room_vacuum)
problem.add_fluent(moving_manipulator)
problem.add_fluent(moving_vacuum)

problem.add_action(move_room_manip)
problem.add_action(move_room_vac)
problem.add_action(tidy_room)
problem.add_action(clean_room)


# =====================
# Map 1
# =====================
def prob1():
    map1 = problem.clone()
    map1.name = 'map1'
    r1 = map1.add_object('r1', room)
    r2 = map1.add_object('r2', room)
    mm1 = map1.add_object('mm1_map1', mobile_manipulator)
    mv1 = map1.add_object('mv1_map1', mobile_vacuum)

    rooms = [r1, r2]

    for r in rooms:
        map1.set_initial_value(clean(r), False)
        map1.set_initial_value(tidy(r), False)
        map1.set_initial_value(current_room_manipulator(mm1, r), False)
        map1.set_initial_value(current_room_vacuum(mv1, r), False)
        for r_other in rooms:
            map1.set_initial_value(connected(r, r_other), False)

    # Moving fluents start False
    map1.set_initial_value(moving_manipulator(mm1), False)
    map1.set_initial_value(moving_vacuum(mv1), False)

    # Connectivity (bidirectional)
    map1.set_initial_value(connected(r1, r2), True)
    map1.set_initial_value(connected(r2, r1), True)

    # Initial robot locations
    map1.set_initial_value(current_room_manipulator(mm1, r1), True)
    map1.set_initial_value(current_room_vacuum(mv1, r1), True)

    for r in rooms:
        map1.add_goal(clean(r))
        map1.add_goal(tidy(r))

    return map1


# =====================
# Map 2
# =====================
def prob2():
    map2 = problem.clone()
    map2.name = 'map2'

    nw = map2.add_object('nw', room)
    ne = map2.add_object('ne', room)
    sw = map2.add_object('sw', room)
    se = map2.add_object('se', room)
    mm2 = map2.add_object('mm1_map2', mobile_manipulator)
    mv2 = map2.add_object('mv1_map2', mobile_vacuum)

    rooms = [nw, ne, sw, se]

    for r in rooms:
        map2.set_initial_value(clean(r), False)
        map2.set_initial_value(tidy(r), False)
        map2.set_initial_value(current_room_manipulator(mm2, r), False)
        map2.set_initial_value(current_room_vacuum(mv2, r), False)
        for r_other in rooms:
            map2.set_initial_value(connected(r, r_other), False)

    # Moving fluents start False
    map2.set_initial_value(moving_manipulator(mm2), False)
    map2.set_initial_value(moving_vacuum(mv2), False)

    # Grid connectivity
    map2.set_initial_value(connected(nw, ne), True)
    map2.set_initial_value(connected(ne, nw), True)
    map2.set_initial_value(connected(nw, sw), True)
    map2.set_initial_value(connected(sw, nw), True)
    map2.set_initial_value(connected(se, ne), True)
    map2.set_initial_value(connected(ne, se), True)
    map2.set_initial_value(connected(se, sw), True)
    map2.set_initial_value(connected(sw, se), True)

    # Initial robot locations
    map2.set_initial_value(current_room_manipulator(mm2, sw), True)
    map2.set_initial_value(current_room_vacuum(mv2, ne), True)

    for r in rooms:
        map2.add_goal(clean(r))
        map2.add_goal(tidy(r))

    return map2


# =====================
# Map 3
# =====================
def prob3():
    map3 = problem.clone()
    map3.name = 'map3'

    rm1 = map3.add_object('rm1', room)
    rm2 = map3.add_object('rm2', room)
    rm3 = map3.add_object('rm3', room)
    rm4 = map3.add_object('rm4', room)
    rm5 = map3.add_object('rm5', room)
    rm6 = map3.add_object('rm6', room)
    rm7 = map3.add_object('rm7', room)
    rm8 = map3.add_object('rm8', room)
    corridor = map3.add_object('corridor', room)

    mm3 = map3.add_object('mm1_map3', mobile_manipulator)
    mv3 = map3.add_object('mv1_map3', mobile_vacuum)

    rooms = [rm1, rm2, rm3, rm4, rm5, rm6, rm7, rm8, corridor]

    for r in rooms:
        map3.set_initial_value(clean(r), False)
        map3.set_initial_value(tidy(r), False)
        map3.set_initial_value(current_room_manipulator(mm3, r), False)
        map3.set_initial_value(current_room_vacuum(mv3, r), False)
        for r_other in rooms:
            map3.set_initial_value(connected(r, r_other), False)

    # Moving fluents start False
    map3.set_initial_value(moving_manipulator(mm3), False)
    map3.set_initial_value(moving_vacuum(mv3), False)

    # Robot starting positions
    map3.set_initial_value(current_room_manipulator(mm3, corridor), True)
    map3.set_initial_value(current_room_vacuum(mv3, corridor), True)

    # Room connectivity (corridor hub)
    for r in [rm1, rm2, rm3, rm4, rm5, rm6, rm7, rm8]:
        map3.set_initial_value(connected(corridor, r), True)
        map3.set_initial_value(connected(r, corridor), True)

    # Initial room states
    for r in [rm1, rm2, rm3, rm4]:
        map3.set_initial_value(tidy(r), True)   # tidy but not clean

    for r in [rm5, rm6, rm7, rm8]:
        map3.set_initial_value(clean(r), True)  # clean but not tidy

    map3.set_initial_value(clean(corridor), True)
    map3.set_initial_value(tidy(corridor), True)

    for r in [rm1, rm2, rm3, rm4, rm5, rm6, rm7, rm8]:
        map3.add_goal(clean(r))
        map3.add_goal(tidy(r))

    return map3


# ====================
# SOLVER
# ====================
from unified_planning.engines import PlanGenerationResultStatus

def solve(prob):
    with OneshotPlanner(name='tamer') as planner:
        result = planner.solve(prob)
        if result.status in [PlanGenerationResultStatus.SOLVED_SATISFICING,
                             PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
            print("SOLVED")
            for start, action, duration in result.plan.timed_actions:
                print(f"{float(start)}: {action} [{float(duration)}]")
        else:
            print("NOT SOLVED")

def solve_aries(prob):
    with OneshotPlanner(name='aries') as planner:
        result = planner.solve(prob)
        if result.status in [PlanGenerationResultStatus.SOLVED_SATISFICING,
                             PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
            print("SOLVED")
            for start, action, duration in result.plan.timed_actions:
                print(f"{float(start)}: {action} [{float(duration)}]")
        else:
            print("NOT SOLVED")


print("Map1")
solve_aries(prob1())

print("\nMap2")
solve_aries(prob2())

print("\nMap3")
solve_aries(prob3())